# Varredura de palavras-chave — protocolo §3 do codebook v2

Este notebook automatiza a **busca**, nunca o **veredito**.

O codebook (§3) exige, para cada um dos 143 documentos, o log de 12 termos:
`experiment` · `A/B` · `randomi[sz]e` · `test` · `trial` · `beta` · `control group` ·
`debrief` · `ethics` · `review board` · `IRB` · `risk assessment` — e diz por quê:
*"é esse log que torna cada No auditável"*. Feito na mão, são ~1.700 buscas.

O que sai daqui é o log preenchido, com cada ocorrência em contexto (±90 caracteres) e
com os falsos positivos conhecidos sinalizados. **Nenhuma célula emite código de variável.**
Decidir se aquele `test` é sentido experimental ou "contest", e se o serviço é V1 nível 2
ou 3, continua sendo do avaliador — a máquina só diz onde olhar.

Três garantias, nessa ordem:

1. **Integridade** — todo documento é conferido contra o `sha256` do manifesto antes de ser
   lido. O que não confere é posto em quarentena e **não entra nas contagens**, para que um
   arquivo ilegível nunca produza um zero que pareça um achado.
2. **Validação** — os padrões são testados contra as âncoras publicadas do piloto e contra
   os falsos positivos documentados, antes de rodar em massa.
3. **Varredura** — só então os 143 documentos, dirigidos pelo manifesto.

## 1. Ambiente

No Colab, a célula abaixo clona os dois repos públicos. Localmente, ela não faz nada.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    for repo in ("experimented-without-consent", "experimented-without-consent-corpus"):
        if not Path(repo).exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", f"https://github.com/mrcsvg/{repo}.git"],
                check=True,
            )
    INSTRUMENT = Path("experimented-without-consent")
    CORPUS = Path("experimented-without-consent-corpus")
else:
    # Rodando de dentro de notebooks/ no repo do instrumento.
    INSTRUMENT = Path.cwd().parent
    CORPUS = INSTRUMENT.parent / "experimented-without-consent-corpus"

sys.path.insert(0, str(INSTRUMENT / "analysis"))
import patterns as P

print(f"instrumento : {INSTRUMENT.resolve()}")
print(f"corpus      : {CORPUS.resolve()}")
print(f"padrões §3  : {len(P.PATTERNS)} termos")

## 2. Integridade do corpus

Cada arquivo do corpus abre com um cabeçalho de procedência (FONTE / CAPTURADO / MÉTODO /
SHA-256) e o hash cobre **o corpo abaixo da régua**, não o arquivo inteiro — o cabeçalho não
poderia conter o hash de si mesmo.

Documento que não confere vai para `QUARENTENA` e sai de todas as contagens seguintes.
Isso é deliberado: um arquivo ilegível varrido normalmente devolveria `experiment:0 /
test:0 / …`, e esse zero entraria no dossiê indistinguível de uma ausência substantiva.

In [ ]:
import hashlib

manifest = P.load_manifest(CORPUS)
print(f"corpus congelado em {manifest['frozen_at']}  ·  vantagem {manifest['vantage']}")

QUARENTENA, INTEGROS = [], {}

for servico, entradas in manifest["services"].items():
    ok = []
    for e in entradas:
        corpo = P.strip_provenance_header((CORPUS / e["file"]).read_text(encoding="utf-8"))
        digest = hashlib.sha256(corpo.encode("utf-8")).hexdigest()
        if digest == e["sha256"]:
            ok.append((e, corpo))
        else:
            lixo = corpo.count("�")
            QUARENTENA.append({
                "servico": servico, "file": e["file"], "url": e["url"],
                "sha256_manifesto": e["sha256"], "sha256_disco": digest,
                "chars": len(corpo), "replacement_chars": lixo,
                "pct_ilegivel": round(100 * lixo / max(len(corpo), 1), 1),
            })
    INTEGROS[servico] = ok

n_ok = sum(len(v) for v in INTEGROS.values())
print(f"\nverificados : {n_ok + len(QUARENTENA)}")
print(f"íntegros    : {n_ok}")
print(f"quarentena  : {len(QUARENTENA)}")

if QUARENTENA:
    print("\n⚠  DOCUMENTOS EM QUARENTENA — não entram nas contagens:\n")
    for q in QUARENTENA:
        print(f"  {q['servico']:14} {Path(q['file']).name[:40]:42} {q['pct_ilegivel']:5.1f}% ilegível")
    print("\n  Serviços com o conjunto vinculante afetado ficam NÃO CODIFICÁVEIS até recaptura.")
    print("  As URLs estão no manifesto; ver analysis/freeze-sources.py.")

## 3. Validação dos padrões

Antes de confiar no log automático, os padrões passam por dois testes.

**Sintético** — strings construídas a partir dos falsos positivos que o próprio codebook
documenta: `test` não pode acender em "contest"; `trial` acende em "free trial" mas tem que
vir **sinalizado**, porque período gratuito de assinatura não é experimentação (§2, V1).

**Contra o corpus** — as âncoras publicadas do piloto. O codebook registra que o Google
nomeia explicitamente experimentos ("700,000 experiments"), então o termo `experiment`
*tem* que acender nos documentos do Google. Se não acender, o padrão está quebrado.

> Os alvos usados aqui são só os que já estão publicados no codebook — o notebook não
> carrega nem revela a codificação da primeira passada.

In [ ]:
falhas = []

def checar(descricao, condicao):
    falhas.append(descricao) if not condicao else None
    print(f"  {'ok  ' if condicao else 'FALHA'} {descricao}")

print("Sintético — deve acender:")
for termo, texto in [
    ("experiment", "we run experiments to improve the service"),
    ("experiment", "our experimentation platform serves variants"),
    ("A-B",        "we perform A/B tests on the checkout flow"),
    ("A-B",        "A-B testing informs ranking changes"),
    ("randomize",  "users are randomised into treatment groups"),
    ("randomize",  "we randomize which layout you see"),
    ("control group", "compared against a control group"),
    ("IRB",        "reviewed by our IRB before launch"),
]:
    counts, _ = P.scan_text(texto)
    checar(f"{termo:15} em {texto[:44]!r}", counts[termo] >= 1)

print("\nSintético — NÃO pode acender (falsos positivos do §3):")
for termo, texto in [
    ("test", "enter our photo contest to win"),
    ("test", "the latest version of the app"),
    ("test", "users may protest a decision"),
    ("IRB",  "the word irb in lowercase prose"),
]:
    counts, _ = P.scan_text(texto)
    checar(f"{termo:15} NÃO em {texto[:40]!r}", counts[termo] == 0)

print("\nSintético — acende mas tem que vir sinalizado para triagem:")
for termo, texto in [
    ("trial", "start your 30-day free trial today"),
    ("beta",  "join the beta program for early access"),
]:
    counts, hits = P.scan_text(texto)
    sinalizado = counts[termo] >= 1 and any(h.term == termo and h.flag for h in hits)
    checar(f"{termo:15} sinalizado em {texto[:36]!r}", sinalizado)

def resolver(prefixo):
    """Resolve o nome longo do serviço a partir do prefixo, sem cair em vizinho.

    Os nomes no manifesto carregam provedor e tipo DSA ('Google Search (Google
    Ireland Ltd.; DSA VLOSE)'). Casar por substring solta faria 'Google' bater em
    Google Maps — e a âncora passaria mesmo com os documentos do Search quebrados.
    """
    achados = [s for s in INTEGROS if s.startswith(prefixo)]
    if len(achados) != 1:
        raise KeyError(f"{prefixo!r} resolveu para {achados or 'nada'} — esperava exatamente 1")
    return achados[0]

print("\nContra o corpus — âncoras publicadas do codebook:")
for prefixo, termo in [("Google Search", "experiment"), ("Facebook", "test")]:
    servico = resolver(prefixo)
    total = sum(P.scan_text(corpo)[0][termo] for _, corpo in INTEGROS[servico])
    checar(f"{termo:15} acende em {servico[:38]} ({total} hits)", total >= 1)

print()
if falhas:
    raise AssertionError(f"{len(falhas)} validação(ões) falharam — não rode a varredura assim:\n  "
                         + "\n  ".join(falhas))
print("✓ padrões validados — pode varrer")

## 4. Varredura

Os 143 documentos, dirigidos pelo manifesto (nunca por glob: o repo tem diretórios duplicados do macOS que não são rastreados).

In [ ]:
from collections import defaultdict

dossie = {}
totais = defaultdict(int)

for servico, docs in INTEGROS.items():
    linhas = []
    for e, corpo in docs:
        counts, hits = P.scan_text(corpo)
        for t, c in counts.items():
            totais[t] += c
        linhas.append(P.DocumentHits(
            file=e["file"], url=e["url"], sha256=e["sha256"],
            chars=e["chars"], counts=counts, hits=hits,
        ))
    dossie[servico] = linhas

print(f"{'termo':18} {'hits':>6}   documentos com ≥1")
for t in P.PATTERNS:
    ndocs = sum(1 for docs in dossie.values() for d in docs if d.counts[t])
    print(f"{t:18} {totais[t]:6}   {ndocs:3} de {sum(len(v) for v in dossie.values())}")

In [ ]:
import json, datetime

destino = INSTRUMENT / "analysis" / "keyword-hits.json"
saida = {
    "gerado_por": "notebooks/01-keyword-sweep.ipynb",
    "corpus_frozen_at": manifest["frozen_at"],
    "vantagem": manifest["vantage"],
    "termos": {t: P.PATTERNS[t]["regex"] for t in P.PATTERNS},
    "quarentena": QUARENTENA,
    "servicos": {s: [d.to_dict() for d in docs] for s, docs in dossie.items()},
}
destino.write_text(json.dumps(saida, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"dossiê gravado: {destino}  ({destino.stat().st_size/1024:.0f} KB)")
print("\nÉ este arquivo que o instrumento consome para pré-preencher o campo keyword_log.")

## 5. Conferindo um serviço

É assim que o dossiê chega na tela do avaliador: a linha de log pronta e cada ocorrência em contexto, com a sinalização de triagem quando existe.

In [ ]:
SERVICO = "Booking.com"   # troque à vontade

for d in dossie[SERVICO]:
    print(f"\n── {Path(d.file).name}")
    print(f"   {d.url[:96]}")
    print(f"   log: {d.log_line}")
    for h in d.hits[:6]:
        print(f"     [{h.term}] {h.kwic[:150]}")
        if h.flag:
            print(f"        ⚠ {h.flag}")
    if len(d.hits) > 6:
        print(f"     … mais {len(d.hits) - 6} ocorrência(s)")

## O que este notebook **não** faz

Não sugere código, não pontua, não decide. A triagem — este `test` é experimental ou é
"contest"? este trecho é V1 nível 2 ou 3? em que registro ele vive? — é do avaliador, e é
justamente ela que o codebook pede que seja humana.

O limite conhecido: regex não pega paráfrase. Um documento que diga *"sometimes we show
different people different designs to find out how they react"* tem **zero** dos 12 termos
e é reconhecimento explícito de experimentação. Esse buraco é o que o
`02-llm-recall-sweep.ipynb` mede — e medir o buraco é resultado de método, não muleta.